In [1]:
import pandas as pd

columns = [
    "transaction_id", "price", "date_of_transfer", "postcode",
    "property_type", "old_new", "duration", "paon", "saon",
    "street", "locality", "town_city", "district", "county",
    "ppd_category", "record_status"
]

df = pd.read_csv("../data/raw/pp-2024.csv", names=columns, header=None)

In [2]:
df.shape

(927623, 16)

In [3]:
df.head()

,transaction_id,price,date_of_transfer,postcode,property_type,old_new,duration,paon,saon,street,locality,town_city,district,county,ppd_category,record_status
0,{25E9DA80-AD30-555E-E063-4704A8C066F2},225000,2024-10-02 00:00,DE6 1TW,S,N,F,48,NaN,ACORN DRIVE,NaN,ASHBOURNE,DERBYSHIRE DALES,DERBYSHIRE,A,A
1,{25E9DA80-AD31-555E-E063-4704A8C066F2},120000,2024-10-04 00:00,SK22 4AH,F,N,L,8,NaN,MEAL STREET,NEW MILLS,HIGH PEAK,HIGH PEAK,DERBYSHIRE,A,A
2,{25E9DA80-AD32-555E-E063-4704A8C066F2},197500,2024-08-19 00:00,S42 5FN,T,N,F,24,NaN,FARMHOUSE WAY,GRASSMOOR,CHESTERFIELD,NORTH EAST DERBYSHIRE,DERBYSHIRE,A,A
3,{25E9DA80-AD33-555E-E063-4704A8C066F2},275000,2024-07-17 00:00,S40 3HF,D,N,F,22,NaN,GREENWAYS,NaN,CHESTERFIELD,CHESTERFIELD,DERBYSHIRE,A,A
4,{25E9DA80-AD34-555E-E063-4704A8C066F2},216000,2024-02-09 00:00,DE24 3GP,S,N,F,7,NaN,LOWICK CLOSE,NaN,DERBY,CITY OF DERBY,CITY OF DERBY,A,A


In [4]:
df.dtypes

transaction_id        str
price               int64
date_of_transfer      str
postcode              str
property_type         str
old_new               str
duration              str
paon                  str
saon                  str
street                str
locality              str
town_city             str
district              str
county                str
ppd_category          str
record_status         str
dtype: object

In [5]:
target_postcode = "DE6 1TW"
matches = df[df["postcode"] == target_postcode]
matches

,transaction_id,price,date_of_transfer,postcode,property_type,old_new,duration,paon,saon,street,locality,town_city,district,county,ppd_category,record_status
0,{25E9DA80-AD30-555E-E063-4704A8C066F2},225000,2024-10-02 00:00,DE6 1TW,S,N,F,48,NaN,ACORN DRIVE,NaN,ASHBOURNE,DERBYSHIRE DALES,DERBYSHIRE,A,A
587494,{237B17FD-DB86-22AC-E063-4804A8C0EA3A},380000,2024-05-03 00:00,DE6 1TW,D,N,F,23,NaN,ACORN DRIVE,NaN,ASHBOURNE,DERBYSHIRE DALES,DERBYSHIRE,A,A


In [6]:
target_district = target_postcode.split(" ")[0]  # "DE6"

district_matches = df[df["postcode"].str.startswith(target_district + " ")]
district_matches.shape

(427, 16)

In [7]:
district_matches["date_of_transfer"] = pd.to_datetime(district_matches["date_of_transfer"])
district_matches[["date_of_transfer", "price"]].sort_values("date_of_transfer").head()

,date_of_transfer,price
721707,2024-01-04,1100000
349640,2024-01-05,449995
684173,2024-01-05,375000
439393,2024-01-09,250000
818088,2024-01-10,160000


In [8]:
earliest_date = district_matches["date_of_transfer"].min()
latest_date = district_matches["date_of_transfer"].max()

print("Earliest:", earliest_date)
print("Latest:", latest_date)

# Average price in the first 3 months of data vs last 3 months
start_price = district_matches[district_matches["date_of_transfer"] <= earliest_date + pd.Timedelta(days=90)]["price"].mean()
end_price = district_matches[district_matches["date_of_transfer"] >= latest_date - pd.Timedelta(days=90)]["price"].mean()

print("Start avg price:", start_price)
print("End avg price:", end_price)

Earliest: 2024-01-04 00:00:00
Latest: 2024-12-20 00:00:00
Start avg price: 393420.1568627451
End avg price: 443618.88


In [9]:
years_diff = (latest_date - earliest_date).days / 365.25

if years_diff > 0 and start_price > 0:
    cagr = (end_price / start_price) ** (1 / years_diff) - 1
    print(f"CAGR: {cagr * 100:.2f}% per year")
else:
    print("Not enough date range to calculate CAGR")

CAGR: 13.31% per year


In [10]:
property_price = 225000      # from our DE6 1TW example
monthly_rent = 950            # example estimate
estimated_annual_costs = 1500 # e.g., insurance, maintenance reserve

annual_rent = monthly_rent * 12

gross_yield = (annual_rent / property_price) * 100
net_yield = ((annual_rent - estimated_annual_costs) / property_price) * 100

print(f"Annual rent: £{annual_rent}")
print(f"Gross Rental Yield: {gross_yield:.2f}%")
print(f"Net Rental Yield: {net_yield:.2f}%")

Annual rent: £11400
Gross Rental Yield: 5.07%
Net Rental Yield: 4.40%
